##### Ingest circuits.csv file

##### Read csv file using spark dataframe reader

In [0]:
%fs ls /databricks-datasets/

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")


In [0]:
dbutils.widgets.text("p_file_date", "2021-03-21")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
circuits_df = spark.read.option("header", True).csv(f"{raw_path}/circuits.csv")

In [0]:
circuits_schema = StructType(fields=[StructField("circuitId", IntegerType(), False),
                                     StructField("circuitRef", StringType(), True),
                                     StructField("name", StringType(), True),
                                     StructField("location", StringType(), True),
                                     StructField("country", StringType(), True) ,
                                     StructField("lat", DoubleType(), True),
                                     StructField("lng", DoubleType(), True),
                                     StructField("alt", IntegerType(), True),
                                     StructField("url", StringType(), True)])

In [0]:
circuits_df = spark.read \
.schema(circuits_schema) \
.csv(f"{raw_path}/{v_file_date}/circuits.csv")

##### Select only the required columns

In [0]:
circuits_selected_df = circuits_df.select(col("circuitId"), col("circuitRef"), col("name"), col("location"), col("country"), col("lat"), col("lng"), col("alt"))

##### Rename the columns as required

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnRenamed("circuitId", "circuit_id") \
.withColumnRenamed("circuitRef", "circuit_ref") \
.withColumnRenamed("lat", "latitude") \
.withColumnRenamed("lng", "longitude") \
.withColumnRenamed("alt", "altitude") \
.withColumn("data_source", lit(v_data_source)) \
.withColumn("file_date", lit(v_file_date))

##### Add ingestion date to the dataframe

In [0]:
circuits_final_df = add_ingestion_date(circuits_renamed_df)

###### Write data to datalake as parquet

In [0]:
circuits_final_df.write.mode("overwrite").parquet(f"{processed_path}/circuits")

###### Create external table

In [0]:
# circuits_final_df.write.mode("overwrite").format("parquet").saveAsTable(f"f1_processed.circuits")

##### Check

In [0]:
df = spark.read.parquet(f"{processed_path}/circuits")
display(df)

In [0]:
dbutils.notebook.exit("SUCCESS")